### Paper classification based on a model trained with relevant and non-relevant data

In [ ]:
# Import required libraries for data handling, model training, and evaluation
import pandas as pd

from datasets import Dataset, DatasetDict
from SetFitTrainer, Trainer, TrainingArguments
from sentence_transformers.losses import CosineSimilarityLoss
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# Configure the dataset paths, column names, and base model settings
project_path = ""

train_file = f"{project_path}rel_nrel.csv"
label_column = "label"
title_column = "title"
abstract_column = "abstract"

#base_model = "BAAI/bge-small-en-v1.5"
base_model = "all-MiniLM-L6-v2"

saved_model = f"{project_path}rel-nrel-100perc-all-MiniLM-L6-v2"

In [ ]:
# Load training CSV, clean missing fields, and create combined text input

df = pd.read_csv(train_file)
df[abstract_column] = df[abstract_column].fillna("")
df[title_column] = df[title_column].fillna("")
df['label'] = df[label_column]
df['text'] = df[title_column] + '.\n' + df[abstract_column]

# Split into train and test sets for evaluation
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

ds = DatasetDict()

ds['train'] = Dataset.from_pandas(df_train)
ds['eval'] = Dataset.from_pandas(df_test)

In [ ]:
# Print training and evaluation dataset sizes for each class
print(f"Relevant training size: {len(df_train[df_train['label'] == 'relevant'])}")
print(f"Non-relevant training size: {len(df_train[df_train['label'] != 'relevant'])}")
print(f"Relevant test size: {len(df_test[df_test['label'] == 'relevant'])}")
print(f"Non-relevant test size: {len(df_test[df_test['label'] != 'relevant'])}")

In [ ]:
# Load the pretrained SetFit model and configure training arguments
model = SetFitModel.from_pretrained(base_model)

args = TrainingArguments(
    batch_size=32, # 32,
    num_iterations=5,
    num_epochs=10,
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds['train'],
    eval_dataset=ds["eval"]
)

In [ ]:
trainer.train()

In [ ]:
# Save the fine-tuned model for future reuse
model.save_pretrained(saved_model)

In [ ]:
# Evaluate model performance on the held-out evaluation dataset
eval_texts = ds["eval"]["text"]
true_labels = ds["eval"]["label"]

pred_labels = model.predict(eval_texts)

accuracy = accuracy_score(true_labels, pred_labels)
print(f"Accuracy: {accuracy:.2f}")

print("\nClassification Report:")
print(classification_report(true_labels, pred_labels))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, pred_labels))

In [ ]:
# Save evaluation dataset with predicted labels for later analysis
results = pd.DataFrame(ds["eval"])
results["pred"] = pred_labels
results.to_csv("results_100perc_data.csv")

### Making predictions on the remaining data

In [ ]:
# Load the new dataset and keep only relevant metadata fields
data_file = f"{project_path}bibfile_data_40k.pkl"

data = pd.read_pickle(data_file)
data = data[["doi", "title", "abstract"]]

data

In [ ]:
# Predict relevance labels for the new data abstracts
pred_texts = data["abstract"]

pred_labels = model.predict(pred_texts, show_progress_bar=True)

In [ ]:
# Store prediction results and create separate CSV files for relevant and non-relevant items
data["predicted"] = pred_labels
data.to_csv(f"{project_path}data_predicted_rel_notrel.csv")
data.to_pickle(f"{project_path}data_predicted_rel_notrel.pkl")

rel = data[data["predicted"] == "relevant"]
rel.to_csv(f"{project_path}data_predicted_relevant.csv")
not_rel = data[data["predicted"] != "relevant"]
not_rel.to_csv(f"{project_path}data_predicted_not_relelevant.csv")

print(f"Relevant items: {len(rel)}")
print(f"Non-relevant items: {len(not_rel)}")
